# Connect to DB

In [1]:
import re
import sys
import os
from loguru import logger

from pymongo import MongoClient

# PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))))
PROJECT_ROOT = "/home/ubuntu/projects/AI/git/dev/v03/law-document-sync-core-service"
sys.path.append(PROJECT_ROOT)
from constants import MongoDBConfig, MigrateConfig, MongoDBCollectionConfig


client = MongoClient(host=MongoDBConfig.HOST, 
                    port=MongoDBConfig.PORT,
                    username=MongoDBConfig.USERNAME,
                    password=MongoDBConfig.PASSWORD)

db = client[MigrateConfig.MIGRATE_CORE_DB]
documents_collection = db[MongoDBCollectionConfig.DOCUMENT_COLLECTION_NAME]
reference_collection = db[MongoDBCollectionConfig.LAW_REFERENCE_COLLECTION_NAME]

# Phân tích dữ liệu

In [ ]:
references = list(reference_collection.find({'source_type': 'DOCUMENT', 'target_type': 'DOCUMENT'}))

Number of References: 3621241


In [14]:
# Liệt kê các giá trị trường reference_type
reference_collection.distinct('reference_type')

['AMEND',
 'AMENDED',
 'BASIS',
 'CONSOLIDATED',
 'CONTENT_CONNECTION',
 'DETAIL',
 'REPLACE',
 'REPLACED']

In [16]:
references[0]

{'_id': ObjectId('680c7e5fc9b0edc6b7e236e8'),
 'reference_id': '8b8f3016-2bdf-4dd8-88c3-9705f8f282e2',
 'source_id': '635289',
 'source_type': 'DOCUMENT',
 'target_id': '524363',
 'target_type': 'DOCUMENT',
 'reference_status': 'Còn hiệu lực',
 'reference_type': 'AMENDED',
 'created_date': datetime.datetime(2025, 3, 7, 16, 27, 38),
 'last_modified': '13:34:07 26/04/25',
 'last_modified_by': ''}

# Định nghĩa hàm tìm kiếm

In [77]:
def get_documents_info(doc_id):
    documents = documents_collection.find({"doc_id": doc_id}, {"doc_code": 1, "doc_title": 2})
    return list(documents)

In [100]:
def get_documents_related(doc_id):    
    
    amended_doc_ids= set()
    replaced_doc_ids= set()
    B_doc_ids = set()
    
    references = reference_collection.find({"source_id": doc_id, "reference_type": {"$in": ["AMENDED", "REPLACED"]}})
    for reference in list(references):
        if reference['reference_type'] == "AMENDED":
            amended_doc_ids.add(reference['target_id'])
        elif reference['reference_type'] == "REPLACED":
            replaced_doc_ids.add(reference['target_id'])
        B_doc_ids.add(reference['target_id'])

        
    A_doc_ids = set()
    C_doc_ids = set() 
    if B_doc_ids:
        for doc_id in list(B_doc_ids):            
            references = reference_collection.find({"source_id": doc_id, "reference_type": {"$in": ["BASIS", "AMEND", "REPLACE", "DETAIL", "CONTENT_CONNECTION"]}})
            for reference in list(references):
                if reference['reference_type'] == "BASIS":
                    C_doc_ids.add(reference['target_id'])
                elif reference['target_id'] not in C_doc_ids not in B_doc_ids:
                    A_doc_ids.add(reference['target_id'])
        
    
    D_doc_ids = set()
    references = reference_collection.find({"source_id": doc_id, "reference_type": "DETAIL"})
    for reference in list(references):
        D_doc_ids.add(reference['target_id'])

    E_doc_ids = set()
    references = reference_collection.find({"source_id": doc_id, "reference_type": "CONTENT_CONNECTION"})
    for reference in list(references):
        E_doc_ids.add(reference['target_id'])

    
    return {
        "A_doc_ids": A_doc_ids,
        "B_doc_ids": B_doc_ids,
        "C_doc_ids": C_doc_ids,
        "D_doc_ids": D_doc_ids,
        "E_doc_ids": E_doc_ids,
    }   

In [101]:
doc_id = "296661"
related_docs = get_documents_related(doc_id)

In [104]:
def filter_documents(related_docs):
    final_doc_ids = set()
    for _, doc_ids in related_docs.items():
        final_doc_ids.update(doc_ids)
    return final_doc_ids    

In [ ]:
for type_relationship, doc_ids in related_docs.items():
    for doc_id in doc_ids:
        doc_info = get_documents_info(doc_id)

A_doc_ids
[{'_id': ObjectId('680cb428050e68f844fbe549'), 'doc_code': '02/2010/NQ-HĐTP', 'doc_title': 'Nghị quyết 02/2010/NQ-HĐTP bổ sung hướng dẫn của Nghị quyết 01/2007/NQ-HĐTP và Nghị quyết 02/2007/NQ-HĐTP do Hội đồng thẩm phán Toà án nhân dân tối cao ban hành'}]
[{'_id': ObjectId('680cc932050e68f84403690a'), 'doc_code': '100/2015/QH13', 'doc_title': 'Bộ luật hình sự 2015'}]
[{'_id': ObjectId('680cc043050e68f844005c50'), 'doc_code': '01/2000/NQ-HĐTP', 'doc_title': 'Nghị quyết 01/2000/NQ-HĐTP về hướng dẫn áp dụng một số quy định trong phần chung của Bộ Luật Hình sự năm 1999 do Hội đồng thẩm phán toà án nhân dân tối cao ban hành'}]
[{'_id': ObjectId('680cc0c9050e68f844008cb4'), 'doc_code': '001/SLT', 'doc_title': 'Sắc luật số 001/SLT về việc cấm chỉ mọi hành động đầu cơ về kinh tế do Chủ tịch nước ban hành'}]
[{'_id': ObjectId('680cceb3050e68f84405349c'), 'doc_code': '72/2010/NĐ-CP', 'doc_title': 'Nghị định 72/2010/NĐ-CP quy định về phòng ngừa, đấu tranh chống tội phạm và vi phạm pháp 

In [ ]:
filtered_doc_ids = filter_documents(related_docs=related_docs)

A_doc_ids: 51
B_doc_ids: 2
C_doc_ids: 2
D_doc_ids: 6
E_doc_ids: 0
54
